In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("1. 준비된 AI 학습용 데이터셋 불러오기...")
df = pd.read_csv("ML_Ready_Dataset.csv", index_col=0)

# X (시험과목): 30개 유전자 수치 데이터 (Label 열만 뺌)
X = df.drop('Label', axis=1)
# y (정답지): 암(1)인지 정상(0)인지
y = df['Label']

print("2. 데이터 반갈죽 (학습용 70% vs 실전 모의고사용 30%) 진행 중...")
# 589명의 환자를 무작위로 섞어서 나눕니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f" -> AI가 공부할 환자 수: {len(X_train)}명")
print(f" -> 모의고사에 쓸 처음 보는 환자 수: {len(X_test)}명\n")

print("3. 인공지능 모델(Random Forest) 소환 및 학습 시작! 🧠")
# n_estimators=100: "100명의 의사(의사결정나무)를 모아서 다수결 투표로 진단하겠다"는 뜻입니다.
ai_model = RandomForestClassifier(n_estimators=100, random_state=42)

# .fit() 이 단어 하나가 바로 '학습해라!' 라는 마법의 명령어입니다.
ai_model.fit(X_train, y_train)
print(" -> 삐빅. 학습 완료.\n")

print("4. 모의고사(Test 데이터) 채점 및 확률(%) 계산 중...")
# 1) 단순히 '암이다/아니다' 예측하기
predictions = ai_model.predict(X_test)

# 2) "암일 확률이 몇 %인가?" 예측하기 (우리의 최종 목표!)
# predict_proba는 [정상일 확률, 암일 확률] 두 개를 뱉어내므로, 두 번째 값([:, 1])만 가져옵니다.
probabilities = ai_model.predict_proba(X_test)[:, 1]

print("==============================================================")
print("🏆 [AI 모의고사 채점 결과]")
print("==============================================================")
# 최종 정확도 채점
accuracy = accuracy_score(y_test, predictions)
print(f"✅ 실전 진단 정확도: {accuracy * 100:.2f}%\n")

print("[상세 성적표]")
print(classification_report(y_test, predictions, target_names=['정상(0)', '폐선암(1)']))

# 처음 5명의 환자에 대해 AI가 진단한 '확률(%)' 훔쳐보기
print("\n[새로운 환자 5명 진단 시뮬레이션]")
for i in range(5):
    actual = "암" if y_test.iloc[i] == 1 else "정상"
    pred_prob = probabilities[i] * 100
    print(f"환자 {i+1} (실제: {actual}) -> AI 진단: 폐선암일 확률 {pred_prob:.1f}%")

1. 준비된 AI 학습용 데이터셋 불러오기...
2. 데이터 반갈죽 (학습용 70% vs 실전 모의고사용 30%) 진행 중...
 -> AI가 공부할 환자 수: 412명
 -> 모의고사에 쓸 처음 보는 환자 수: 177명

3. 인공지능 모델(Random Forest) 소환 및 학습 시작! 🧠
 -> 삐빅. 학습 완료.

4. 모의고사(Test 데이터) 채점 및 확률(%) 계산 중...
🏆 [AI 모의고사 채점 결과]
✅ 실전 진단 정확도: 100.00%

[상세 성적표]
              precision    recall  f1-score   support

       정상(0)       1.00      1.00      1.00        14
      폐선암(1)       1.00      1.00      1.00       163

    accuracy                           1.00       177
   macro avg       1.00      1.00      1.00       177
weighted avg       1.00      1.00      1.00       177


[새로운 환자 5명 진단 시뮬레이션]
환자 1 (실제: 암) -> AI 진단: 폐선암일 확률 100.0%
환자 2 (실제: 암) -> AI 진단: 폐선암일 확률 100.0%
환자 3 (실제: 암) -> AI 진단: 폐선암일 확률 100.0%
환자 4 (실제: 암) -> AI 진단: 폐선암일 확률 86.0%
환자 5 (실제: 암) -> AI 진단: 폐선암일 확률 98.0%


In [4]:
import joblib

print("\n5. 최종 진단 시스템 구축 중...")
# 1) 훈련된 AI 모델을 영구 파일로 저장하기 (이제 파이썬을 꺼도 AI의 기억이 유지됩니다!)
joblib.dump(ai_model, 'LUAD_Prediction_AI_Model.pkl')
print("✅ AI 모델의 '뇌'가 저장되었습니다! (LUAD_Prediction_AI_Model.pkl)")

# ---------------------------------------------------------
# (시간이 흘러... 완전히 새로운 환자의 유전자 검사 결과가 도착했다고 가정해봅시다)
# ---------------------------------------------------------

print("\n6. 새로운 가상 환자 진단 시뮬레이션 시작...")
# 2) 저장된 AI 모델 다시 불러오기
loaded_ai = joblib.load('LUAD_Prediction_AI_Model.pkl')

# 3) 새로운 환자의 데이터 입력 (예시: 실전 데이터 중 하나를 몰래 복사해옵니다)
# 실제 웹서비스를 만들 때는 '새환자_유전자.csv' 파일을 pd.read_csv()로 불러오게 됩니다.
new_patient_data = X_test.iloc[[0]].copy() 
new_patient_data.index = ['New_Patient_Kangsan_001'] # 가상의 환자 이름 부여

# 4) AI 진단 시작!
# predict_proba로 확률을 계산하고, 0번째(첫 번째 환자)의 1번 위치(암 확률) 값을 가져옵니다.
probability = loaded_ai.predict_proba(new_patient_data)[:, 1][0]

# 확률이 50%(0.5)를 넘으면 암으로 판정하는 로직
diagnosis = "폐선암(LUAD) 고위험군" if probability > 0.5 else "정상 소견"

print("\n========================================")
print("🩺 [초정밀 AI 유전자 진단 결과 보고서]")
print("========================================")
print(f"환자 ID: {new_patient_data.index[0]}")
print(f"분석 모델: 30-Gene Feature Random Forest")
print(f"AI 판독 결과: {diagnosis}")
print(f"폐선암 발병 확률: {probability * 100:.1f}%")
print("========================================")


5. 최종 진단 시스템 구축 중...
✅ AI 모델의 '뇌'가 저장되었습니다! (LUAD_Prediction_AI_Model.pkl)

6. 새로운 가상 환자 진단 시뮬레이션 시작...

🩺 [초정밀 AI 유전자 진단 결과 보고서]
환자 ID: New_Patient_Kangsan_001
분석 모델: 30-Gene Feature Random Forest
AI 판독 결과: 폐선암(LUAD) 고위험군
폐선암 발병 확률: 100.0%
